# Audio Filtering Pipeline for Indic Speech
**Assignment — Sarvam AI**

Automatically detect and remove low-quality audio samples from large multilingual Indic speech datasets before training speech models (ASR / TTS).

**Dataset:** [ai4bharat/IndicVoices](https://huggingface.co/datasets/ai4bharat/IndicVoices) — 23.7K hours, 22 Indic languages, CC-BY-4.0

> **Pre-requisite:** Request dataset access at https://huggingface.co/datasets/ai4bharat/IndicVoices  
> Then add your HF token in **Cell 1** below.

**Run all cells in order:** Runtime → Run all

---
## Cell 1 — HuggingFace Login + Download Repository

In [ ]:
import os, sys, zipfile, urllib.request

# ── HuggingFace token ─────────────────────────────────────────────────────────
# Option A: Colab Secrets (Recommended — Runtime → Manage secrets → add HF_TOKEN)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("HF token loaded from Colab secrets.")
except Exception:
    # Option B: Paste your token directly (remove before sharing)
    HF_TOKEN = ""   # ← paste your token here if not using Colab secrets

if not HF_TOKEN:
    print("WARNING: No HF token found.")
    print("  → Get your token at https://huggingface.co/settings/tokens")
    print("  → Add it as HF_TOKEN in Colab secrets (Runtime → Manage secrets)")
else:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print(f"HF token set ({len(HF_TOKEN)} chars).")

# ── Download repository ───────────────────────────────────────────────────────
REPO_DIR = "/content/Sarvam-Project"
ZIP_URL  = "https://github.com/Kartikgc9/Sarvam-Project/archive/refs/heads/main.zip"

if not os.path.exists(REPO_DIR):
    print("Downloading repository...")
    urllib.request.urlretrieve(ZIP_URL, "/content/repo.zip")
    with zipfile.ZipFile("/content/repo.zip") as z:
        z.extractall("/content/")
    os.rename("/content/Sarvam-Project-main", REPO_DIR)
    os.remove("/content/repo.zip")
    print("Repository downloaded.")
else:
    print("Repository already present.")

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"Working dir: {os.getcwd()}")
print(f"Contents   : {sorted(os.listdir('.'))}")

---
## Cell 2 — Install Dependencies

In [ ]:
import subprocess, sys

print("Installing dependencies (this takes ~2 min)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "librosa>=0.10.0", "soundfile>=0.12.1", "numpy>=1.24.0",
     "torch>=2.0.0", "onnxruntime>=1.16.0",
     "datasets>=2.14.0", "huggingface_hub>=0.16.0",
     "omegaconf>=2.3.0", "pandas>=2.0.0",
     "matplotlib>=3.7.0", "seaborn>=0.12.0",
     "tqdm>=4.65.0"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("WARNING during install:", result.stderr[-800:])
else:
    print("All dependencies installed OK.")

---
## Cell 3 — Download Models (DNSMOS + Silero-VAD)

In [ ]:
import os, urllib.request

os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# ── DNSMOS P.835 ONNX model ─────────────────────────────────────────────────
DNSMOS_PATH = "models/sig_bak_ovr.onnx"
DNSMOS_URLS = [
    "https://github.com/microsoft/DNS-Challenge/raw/master/DNSMOS/DNSMOS/sig_bak_ovr.onnx",
    "https://github.com/microsoft/DNS-Challenge/raw/refs/heads/master/DNSMOS/DNSMOS/sig_bak_ovr.onnx",
]

if not os.path.exists(DNSMOS_PATH):
    downloaded = False
    for url in DNSMOS_URLS:
        try:
            print(f"Downloading DNSMOS from GitHub...")
            urllib.request.urlretrieve(url, DNSMOS_PATH)
            size_kb = os.path.getsize(DNSMOS_PATH) // 1024
            if size_kb > 100:
                print(f"  DNSMOS OK ({size_kb} KB)")
                downloaded = True
                break
            os.remove(DNSMOS_PATH)
        except Exception as e:
            print(f"  Attempt failed: {e}")
            if os.path.exists(DNSMOS_PATH):
                os.remove(DNSMOS_PATH)
    if not downloaded:
        print("  DNSMOS unavailable — pipeline will use neutral fallback score (3.0).")
else:
    print(f"DNSMOS already present ({os.path.getsize(DNSMOS_PATH)//1024} KB)")

# ── Silero-VAD (via torch.hub) ───────────────────────────────────────────────
import torch
torch.hub.set_dir("models/")
try:
    print("Loading Silero-VAD...")
    _vad, _ = torch.hub.load(
        "snakers4/silero-vad", "silero_vad",
        trust_repo=True, verbose=False, force_reload=False
    )
    print("  Silero-VAD OK")
    del _vad
except Exception as e:
    print(f"  Silero-VAD warning: {e} — pipeline will use fallback VAD score (0.5).")

print("\nModel setup complete.")

---
## Cell 4 — Import Pipeline Modules

In [ ]:
import sys, os
sys.path.insert(0, '/content/Sarvam-Project')
os.chdir('/content/Sarvam-Project')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

from omegaconf import OmegaConf
from IPython.display import display

from src.preprocess import preprocess_audio
from src.metrics.signal_metrics  import compute_all_signal_metrics
from src.metrics.spectral_metrics import compute_all_spectral_metrics
from src.metrics.vad_metrics  import load_vad_model, compute_vad_ratio
from src.metrics.dnsmos       import load_dnsmos_model, compute_dnsmos
from src.scoring              import make_decision

# Load config (IndicVoices)
CFG = OmegaConf.load('config/config.yaml')
SR  = int(CFG.audio.sample_rate)

# Load neural models once
print("Loading VAD model...")
VAD_MODEL, VAD_UTILS = load_vad_model(str(CFG.models.vad_cache_dir))
print("Loading DNSMOS model...")
DNSMOS_SESSION = load_dnsmos_model(str(CFG.models.dnsmos_path))
print("\nAll modules imported. Ready.")

---
## Cell 5 — Metric Intuition on Synthetic Audio

Before running on real data, let's build intuition by computing every metric on 5 synthetic audio types.

In [ ]:
t = np.linspace(0, 2.0, SR * 2, endpoint=False)
rng = np.random.default_rng(42)

SAMPLES = {
    'Clean Speech' : (0.4*np.sin(2*np.pi*200*t)
                    + 0.2*np.sin(2*np.pi*400*t)
                    + 0.1*np.sin(2*np.pi*800*t)).astype(np.float32),
    'Noisy Speech' : None,
    'Clipped'      : None,
    'Silence'      : np.zeros(SR*2, dtype=np.float32),
    'White Noise'  : (0.3*rng.standard_normal(SR*2)).astype(np.float32),
}
SAMPLES['Noisy Speech'] = (SAMPLES['Clean Speech']
                           + 0.25*rng.standard_normal(SR*2).astype(np.float32))
SAMPLES['Clipped'] = np.clip(SAMPLES['Clean Speech'] * 3.0, -1.0, 1.0)

fig, axes = plt.subplots(5, 1, figsize=(14, 10), sharex=True)
time_ax = np.linspace(0, 2.0, SR*2)

for ax, (name, audio) in zip(axes, SAMPLES.items()):
    sig = compute_all_signal_metrics(audio, SR, CFG)
    ax.plot(time_ax, audio, lw=0.5, color='steelblue')
    ax.set_ylim(-1.3, 1.3)
    ax.set_title(
        f"{name}   SNR={sig['snr_db']:.1f}dB  "
        f"Silence={sig['silence_ratio']:.2f}  "
        f"Clipping={sig['clipping_ratio']:.3f}",
        fontsize=9, loc='left'
    )
axes[-1].set_xlabel('Time (s)')
plt.suptitle('Waveforms and Signal Metrics', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

rows = []
for name, audio in SAMPLES.items():
    sig  = compute_all_signal_metrics(audio, SR, CFG)
    spec = compute_all_spectral_metrics(audio, SR)
    rows.append({'Sample': name, **sig, **spec})

df_synth = pd.DataFrame(rows).set_index('Sample')
cols = ['snr_db','rms_energy','silence_ratio','clipping_ratio','spectral_flatness','zcr']
display(df_synth[cols].round(4).style
        .background_gradient(cmap='RdYlGn', axis=0)
        .set_caption('Green = better quality, Red = worse'))

---
## Cell 6 — VAD and DNSMOS on Synthetic Audio

**Silero-VAD:** Neural model (1 MB) — tells us if real speech is present, not just energy.  
**DNSMOS P.835:** Microsoft's perceptual MOS predictor (ONNX, CPU) — scores 1–5 like a human listener.

In [ ]:
print(f"{'Sample':<22} {'VAD':>6}  {'SIG':>5} {'BAK':>5} {'OVR':>5}")
print('-' * 50)
for name, audio in SAMPLES.items():
    vad = compute_vad_ratio(audio, SR, CFG, VAD_MODEL, VAD_UTILS)
    dns = compute_dnsmos(audio, SR, DNSMOS_SESSION)
    bar = ('█' * int(vad*15)).ljust(15)
    print(f"  {name:<20} {bar} {vad:.2f}  "
          f"{dns['dnsmos_sig']:.2f}  {dns['dnsmos_bak']:.2f}  {dns['dnsmos_ovr']:.2f}")
print()
print("VAD = fraction of audio with speech (0=none, 1=all)")
print("SIG/BAK/OVR = DNSMOS speech/background/overall quality (1=bad, 5=excellent)")

---
## Cell 7 — End-to-End Scoring Walkthrough

Two-stage filtering:
1. **Hard rules** — instant discard for extremes (too short, too long, too much clipping/silence)
2. **Weighted quality score** — threshold 0.60 to KEEP

In [ ]:
rng2 = np.random.default_rng(7)
t4   = np.linspace(0, 4.0, SR*4, endpoint=False)
speech = (
    0.30*np.sin(2*np.pi*150*t4) + 0.20*np.sin(2*np.pi*300*t4) +
    0.10*np.sin(2*np.pi*600*t4) + 0.03*rng2.standard_normal(SR*4)
).astype(np.float32)
speech[:SR//2]  = 0.0
speech[-SR//2:] = 0.0

sample = {
    'id': 'hi_demo_001', 'language': 'hi',
    'audio_array': speech, 'sample_rate': SR,
    'transcript': 'यह एक परीक्षण वाक्य है', 'speaker_id': 'spk_01'
}

print('=' * 58)
print('  End-to-end pipeline walkthrough')
print('=' * 58)

sample  = preprocess_audio(sample, SR)
audio   = sample['audio_array']
print(f'\n[1] Preprocessing')
print(f'    Duration   : {sample["duration"]:.2f} s')
print(f'    Peak amp   : {np.max(np.abs(audio)):.4f}  (peak-normalised)')

sig = compute_all_signal_metrics(audio, SR, CFG)
print(f'\n[2] Signal Metrics')
for k, v in sig.items(): print(f'    {k:<22} {v:.4f}')

spec = compute_all_spectral_metrics(audio, SR)
print(f'\n[3] Spectral Metrics')
for k, v in spec.items(): print(f'    {k:<22} {v:.4f}')

vad = compute_vad_ratio(audio, SR, CFG, VAD_MODEL, VAD_UTILS)
print(f'\n[4] Silero-VAD speech ratio : {vad:.4f}')

dns = compute_dnsmos(audio, SR, DNSMOS_SESSION)
print(f'\n[5] DNSMOS P.835')
for k, v in dns.items(): print(f'    {k:<22} {v:.3f}  (1=bad  5=excellent)')

all_m = {**sig, **spec, 'vad_ratio': vad, **dns, 'duration': sample['duration']}
dec   = make_decision(all_m, CFG)
print(f'\n[6] Final Decision')
print(f'    Weighted score : {dec["quality_score"]:.4f}  (threshold = {CFG.scoring.threshold})')
print(f'    Decision       : {dec["decision"]}')
print(f'    Reason         : {dec["reason"]}')

---
## Cell 8 — Score All 5 Synthetic Types

In [ ]:
print(f"Scoring synthetic samples  (threshold = {CFG.scoring.threshold})")
print('=' * 70)

for name, audio in SAMPLES.items():
    s = preprocess_audio(
        {'id': name, 'language': 'demo', 'audio_array': audio.copy(),
         'sample_rate': SR, 'transcript': '', 'speaker_id': 'x'}, SR
    )
    a = s['audio_array']
    all_m = {
        **compute_all_signal_metrics(a, SR, CFG),
        **compute_all_spectral_metrics(a, SR),
        'vad_ratio'   : compute_vad_ratio(a, SR, CFG, VAD_MODEL, VAD_UTILS),
        **compute_dnsmos(a, SR, DNSMOS_SESSION),
        'duration'    : s['duration'],
    }
    dec  = make_decision(all_m, CFG)
    mark = 'KEEP   ' if dec['decision'] == 'KEEP' else 'DISCARD'
    print(f"  {mark}  {name:<22}  score={dec['quality_score']:.3f}  [{dec['reason']}]")

---
## Cell 9 — Run Full Pipeline on IndicVoices

Streams directly from HuggingFace Hub — no 14 GB download needed.  
Processes **`max_samples_per_language` × languages** samples in parallel.

> Requires HF token + approved access (set in Cell 1).

In [ ]:
import time, csv, logging
from concurrent.futures import ProcessPoolExecutor, as_completed

from src.loader    import IndicVoicesLoader
from src.preprocess import preprocess_audio
from src.pipeline  import _compute_cpu_metrics, CSV_COLUMNS

logging.basicConfig(level=logging.WARNING)

OUTPUT_CSV = 'outputs/results.csv'
os.makedirs('outputs', exist_ok=True)

cfg      = OmegaConf.load('config/config.yaml')
cfg_dict = OmegaConf.to_container(cfg, resolve=True)

# Reduce samples for demo (change as needed)
cfg.dataset.max_samples_per_language = 20   # 20 × 10 = 200 samples
cfg.pipeline.num_workers = 2               # Colab free tier: 2 cores

with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    csv.DictWriter(f, fieldnames=CSV_COLUMNS).writeheader()

loader  = IndicVoicesLoader(cfg)
t_start = time.time()
total = kept = discarded = 0

def write_row(result):
    row = {
        'id': result.get('id',''), 'language': result.get('language',''),
        'duration_s'        : round(float(result.get('duration',0)),3),
        'rms_energy'        : round(float(result.get('rms_energy',0)),6),
        'snr_db'            : round(float(result.get('snr_db',0)),2),
        'silence_ratio'     : round(float(result.get('silence_ratio',0)),4),
        'clipping_ratio'    : round(float(result.get('clipping_ratio',0)),6),
        'spectral_flatness' : round(float(result.get('spectral_flatness',0)),4),
        'zcr'               : round(float(result.get('zcr',0)),2),
        'spectral_centroid' : round(float(result.get('spectral_centroid',0)),2),
        'spectral_rolloff'  : round(float(result.get('spectral_rolloff',0)),2),
        'vad_ratio'         : round(float(result.get('vad_ratio',0)),4),
        'dnsmos_sig'        : round(float(result.get('dnsmos_sig',0)),3),
        'dnsmos_bak'        : round(float(result.get('dnsmos_bak',0)),3),
        'dnsmos_ovr'        : round(float(result.get('dnsmos_ovr',0)),3),
        'quality_score'     : round(float(result.get('quality_score',0)),4),
        'decision'          : result.get('decision',''),
        'reason'            : result.get('reason',''),
    }
    with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8') as f:
        csv.DictWriter(f, fieldnames=CSV_COLUMNS).writerow(row)

n_langs   = len(cfg.dataset.languages)
n_samples = cfg.dataset.max_samples_per_language
print(f"Processing {n_samples} samples × {n_langs} languages = {n_samples*n_langs} total...\n")

for batch in loader.iter_batches(int(cfg.pipeline.batch_size)):
    preprocessed = []
    for s in batch:
        try:
            preprocessed.append(preprocess_audio(s, SR))
        except Exception:
            pass

    audio_by_id = {s['id']: (s['audio_array'], s['sample_rate']) for s in preprocessed}

    cpu_results = {}
    with ProcessPoolExecutor(max_workers=int(cfg.pipeline.num_workers)) as ex:
        futs = {ex.submit(_compute_cpu_metrics, s, cfg_dict): s['id'] for s in preprocessed}
        for fut in as_completed(futs):
            try:
                r = fut.result()
                cpu_results[r['id']] = r
            except Exception:
                pass

    for sid, r in cpu_results.items():
        try:
            audio, sr = audio_by_id[sid]
            r['vad_ratio'] = compute_vad_ratio(audio, sr, cfg, VAD_MODEL, VAD_UTILS)
            r.update(compute_dnsmos(audio, sr, DNSMOS_SESSION))
            r.update(make_decision(r, cfg))
            write_row(r)
            if r['decision'] == 'KEEP': kept += 1
            else: discarded += 1
            total += 1
        except Exception:
            pass

    lang = batch[0]['language'] if batch else '?'
    print(f"  [{lang}] batch done — running total: {total} samples "
          f"({kept} kept / {discarded} discarded)")

elapsed = time.time() - t_start
print(f"\nDone in {elapsed:.1f}s")
print(f"Total: {total}  |  Kept: {kept} ({kept/max(total,1)*100:.1f}%)  |  Discarded: {discarded}")
print(f"Results saved to: {OUTPUT_CSV}")

---
## Cell 10 — View Results Table

In [ ]:
df = pd.read_csv('outputs/results.csv')
print(f"Total rows: {len(df)}")
print(f"Kept     : {(df.decision=='KEEP').sum()}  ({(df.decision=='KEEP').mean()*100:.1f}%)")
print(f"Discarded: {(df.decision=='DISCARD').sum()}")
print()
display(df[['id','language','duration_s','snr_db','vad_ratio',
            'dnsmos_ovr','quality_score','decision','reason']].head(20))

---
## Cell 11 — Generate All Plots

In [ ]:
from src.visualize import generate_all_plots
summary = generate_all_plots(
    csv_path='outputs/results.csv',
    output_dir='outputs/plots',
    threshold=float(CFG.scoring.threshold)
)

In [ ]:
from IPython.display import Image, display
import glob

for fpath in sorted(glob.glob('outputs/plots/*.png')):
    print(f"\n{'─'*55}\n  {fpath}\n{'─'*55}")
    display(Image(fpath))

---
## Cell 12 — Per-Language Quality Analysis

In [ ]:
df = pd.read_csv('outputs/results.csv')

lang_stats = (
    df.groupby(['language','decision'])
    .size().unstack(fill_value=0)
    .reindex(columns=['KEEP','DISCARD'], fill_value=0)
)
lang_stats['total']     = lang_stats.sum(axis=1)
lang_stats['keep_rate'] = lang_stats['KEEP'] / lang_stats['total'].clip(lower=1) * 100

print("Keep rate per language:")
display(lang_stats.round(1))

kept_df = df[df['decision']=='KEEP']
if not kept_df.empty:
    avg_m = kept_df.groupby('language')[['snr_db','vad_ratio','dnsmos_ovr','quality_score']].mean().round(3)
    print("\nAverage quality metrics (KEPT samples):")
    display(avg_m)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for ax, (col, title) in zip(axes, [('snr_db','Avg SNR (dB)'),
                                        ('vad_ratio','Avg VAD Ratio'),
                                        ('dnsmos_ovr','Avg DNSMOS OVR'),
                                        ('quality_score','Avg Quality Score')]):
        avg_m[col].plot(kind='bar', ax=ax, color='#3498db', alpha=0.85)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
    plt.suptitle('Quality Metrics by Language (KEPT samples)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## Cell 13 — Scalability Benchmark

In [ ]:
import time

def benchmark_sample(duration_s=3.0):
    audio = (0.4 * np.sin(2*np.pi*200*np.linspace(0, duration_s,
             int(duration_s*SR)))).astype(np.float32)
    t0 = time.perf_counter()
    compute_all_signal_metrics(audio, SR, CFG)
    compute_all_spectral_metrics(audio, SR)
    return time.perf_counter() - t0

times   = [benchmark_sample() for _ in range(20)]
avg_ms  = np.mean(times) * 1000
sps     = 1000 / avg_ms

print(f"CPU metrics per sample  : {avg_ms:.1f} ms")
print(f"Single-core throughput  : {sps:.0f} samples/s")
print(f"2-worker throughput     : ~{sps*2:.0f} samples/s")
print()
print(f"IndicVoices full dataset (~5M samples):")
print(f"  2 workers  → ~{5_000_000/(sps*2)/3600:.0f} hrs")
print(f"  8 workers  → ~{5_000_000/(sps*8)/3600:.0f} hrs")
print(f"  → Use Ray / Spark / Dask for distributed full-scale production")

---
## Cell 14 — Local Run (Optional — after running setup_dataset.py)

If you have already downloaded IndicVoices locally using `setup_dataset.py`, use the `ManifestLoader` to run the pipeline without streaming:

```bash
python setup_dataset.py /path/to/save_dir
```

Then point `ManifestLoader` at the `manifests/` subdirectory.

In [ ]:
# Uncomment and adjust SAVE_DIR after running setup_dataset.py

# from src.loader import ManifestLoader
#
# SAVE_DIR      = "/path/to/save_dir"    # ← set this
# MANIFEST_DIR  = os.path.join(SAVE_DIR, "manifests")
#
# manifest_loader = ManifestLoader(
#     manifest_dir=MANIFEST_DIR,
#     languages=["hi", "ta", "te", "kn", "ml"],   # subset
#     max_per_lang=100,
# )
#
# # Then iterate and process exactly like Cell 9:
# for batch in manifest_loader.iter_batches(batch_size=32):
#     ...   # same processing loop as Cell 9

---
## Summary

| What | How |
|---|---|
| **Dataset** | ai4bharat/IndicVoices — 23.7K hrs, 22 Indic languages, CC-BY-4.0 |
| **Metrics** | Signal (SNR, RMS, clipping, silence) → Spectral (flatness, ZCR) → Neural (Silero-VAD, DNSMOS P.835) |
| **Scoring** | Hard rules first (instant discard), then weighted score ≥ 0.60 |
| **Weights** | SNR 0.30 · VAD 0.25 · DNSMOS 0.20 · active 0.15 · clipping 0.10 |
| **Scalability** | HF streaming (no RAM ceiling) + ProcessPoolExecutor (parallel CPU) + checkpoint recovery |
| **Local option** | `setup_dataset.py` → `ManifestLoader` for offline processing |